# 08W · Does cell state obscure CD4/CD8 lineage in the Adagrasib T cells?

**Start:** Place this notebook beside the working 08V notebook on EC2, restart the **same UV/RAPIDS kernel**, and choose **Run All**. No input H5AD selection or package installation. The default uses all original T cell 1 + T cell 2 cells across the 12 samples. You will get a small **08W_return_to_chat.zip** at the end.

We ask two separate questions: **(1)** Does dropping the least-detected genes before variance scaling preserve the Foxp3 companion signal without fragmenting the graph? **(2)** Does masking selected mitochondrial/ribosomal and then inducible stress genes during PCA reveal a CD4/CD8 axis supported by other transcripts? Neither analysis changes the original counts, broad atlas, cell labels, or QC thresholds. No global regression, cell typing, or extra physical-neighbor calculation.

The ordinary controls come from your completed 08V run. The new embeddings and reports are written separately. The comparison retains the original full-library normalization and ratio weights for genes outside each mask.

In [ ]:
from pathlib import Path
PROJECT_DIR = Path.cwd()  # same original project as 08V; normally no edit
SOURCE_08V_RUN = None    # normally auto-found; optional override: completed run directory containing comparison_complete.json

### Load the packaged code

The collapsed cell restores and checks the embedded sources and sets up diagnostic logs. The notebook works on its own; the ZIP also contains these helpers in readable form.

In [ ]:
import base64, hashlib, importlib, json, os, shutil, sys, time, traceback, zipfile, zlib
from pathlib import Path
from IPython.display import display, FileLink, Image, Markdown
import pandas as pd
_started = time.monotonic()
PLAN = RUN_ROOT = None
PUBLISH_DIR = Path.cwd() / "08W_results"
DIAG = PUBLISH_DIR / "diagnostics" / time.strftime("%Y%m%d_%H%M%S")
DIAG.mkdir(parents=True, exist_ok=True)
_source = zlib.decompress(base64.b64decode('...'))
HELPER_SOURCE_ID = '8f08dfc711559ac02388e953195ea2187bb3a29e06bbe95d20c0b72d529a305e'
if hashlib.sha256(_source).hexdigest() != HELPER_SOURCE_ID:
    raise ValueError("08W notebook payload is incomplete; please download it again.")
HELPER_DIR = (Path.cwd() / ".08W_helpers" / HELPER_SOURCE_ID[:16]).resolve()
HELPER_DIR.mkdir(parents=True,exist_ok=True)
for _name, _s in json.loads(_source).items():
    _dest = HELPER_DIR / _name
    if not _dest.exists() or _dest.read_text(encoding="utf8") != _s:
        _dest.write_text(_s,encoding="utf8")
    sys.modules.pop(Path(_name).stem,None)
sys.path.insert(0,str(HELPER_DIR))
importlib.invalidate_caches()
def show_file(p, label):
    p=Path(p).resolve()
    display(FileLink(os.path.relpath(p,Path.cwd()),result_html_prefix=label+" "))
def diagnostics(exc):
    details="".join(traceback.format_exception(type(exc),exc,exc.__traceback__))
    (DIAG / "exception.txt").write_text(details)
    (DIAG / "context.json").write_text(json.dumps(dict(project_dir=str(PROJECT_DIR), override=str(SOURCE_08V_RUN),
        python=sys.executable, helper_sha=HELPER_SOURCE_ID, run_root=str(RUN_ROOT), plan_id=(PLAN or {}).get("plan_id")),indent=2))
    out=Path.cwd()/"08W_failure_diagnostics.zip"
    with zipfile.ZipFile(out,"w",zipfile.ZIP_DEFLATED) as z:
        roots=[("startup", DIAG)]
        if RUN_ROOT is not None: roots.append(("runtime",Path(RUN_ROOT)))
        size=0
        for group,root in roots:
            if not root.is_dir(): continue
            for path in sorted(root.rglob("*")):
                if not path.is_file() or path.suffix.lower() not in (".log",".json",".txt"): continue
                with path.open("rb") as f:
                    n=path.stat().st_size
                    if n>280_000: f.seek(n-280_000)
                    data=f.read()
                if size+len(data)>7_500_000: continue
                z.writestr(group+"/"+str(path.relative_to(root)),data); size+=len(data)
    print("Return the diagnostic ZIP below. The original 08V run is untouched.")
    show_file(out,"Download diagnostics:")
try:
    import tcell_state_workflow as workflow
    import tcell_state_review as review
except Exception as e:
    diagnostics(e)
    raise
print("08W 1.0.0 | Kernel:",sys.executable,"| helper fingerprint:", HELPER_SOURCE_ID[:16])

## 1 · Find the completed 08V run and audit exactly which genes will change

The selected branch uses the **same frozen 08V panel**, intersects it with detection in ≥500 cells and then scales surviving features to unit variance. The broad branches all start with 08V's static ratio prior (midpoint 0.60). They set weights to zero for named features only. This removes their **direct influence** on distances; correlated programs in other genes can remain. It is intentionally a more limited experiment than regressing a program score out of every gene.

Missing genes do not count as zero expression or an effective intervention. The feature audit records actual coverage. Retaining a matrix column with zero weight and dropping it have the same direct effect on distance; consult effective weights when checking holdouts. Protected programs are monitored, not targeted: exhaustion/checkpoint, IFN, cytotoxic, cycling and canonical lineage/Treg biology.

In [ ]:
try:
    PLAN = workflow.prepare(PROJECT_DIR, source_08v_run=SOURCE_08V_RUN)
    print("08V reference run:", PLAN["source_08v_run"])
    print(f'Cells: {PLAN["n_cells"]:,} | 08W plan: {PLAN["plan_dir"]}')
    class_counts = pd.read_csv(PLAN["feature_class_counts_path"])
    display(class_counts.pivot(index="method",columns="class",values="n_active_genes").fillna(0).astype(int))
    display(pd.DataFrame([{"branch": n,"purpose": b.get("label",n)} for n,b in PLAN["branches"].items()]))
except (Exception, KeyboardInterrupt) as e:
    diagnostics(e)
    raise

## 2 · GPU PCA → Harmony → graph, Leiden and UMAP

| Matched contrast | What it tests |
|---|---|
| 08V selected z-scored vs **selected500_zscore**; and each vs its centered counterpart | Effect of the detection floor, separating it from the effect of scaling |
| 08V broad static ratio vs **broad_no_technical** | Direct contribution from measured mitochondrial-genome and structural cytosolic ribosomal features |
| **broad_no_technical** vs **broad_no_technical_stress** | Additional contribution of a short, fixed early-response/heat-shock panel |
| **broad_ratio_anchorblind** vs **broad_no_technical_stress_anchorblind** | Can the rest of the transcriptome distinguish CD4/CD8 evidence better when neither the coreceptors nor the CD4 companion genes can directly drive the graph? |

The last comparison withholds **Cd4, Cd8a, Cd8b1, Zbtb7b and Cd40lg** from contributing to PCA in both fits. The four original Foxp3 companion holdouts from 08V also remain withheld in every model. The blinded comparison tests the *combined* mask; the two ordinary broad comparisons separate the contributions of its components. These fits use all original rows, including cells with neither coreceptor detected. The 110-minute checkpointed budget applies to model fitting; preflight, preparation and report time are additional.

In [ ]:
try:
    GPU_RUNTIME = workflow.preflight(HELPER_DIR, DIAG / "gpu_preflight")
    BRANCHES, RUN_ROOT = workflow.launch(PLAN, HELPER_DIR, GPU_RUNTIME, max_minutes=110, resume=True)
    print("Completed/verified models:", len(BRANCHES), "| output:", RUN_ROOT)
except (Exception, KeyboardInterrupt) as e:
    RUN_ROOT = getattr(e,"run_root",None) or RUN_ROOT
    diagnostics(e)
    raise

## 3 · Read the lineage, state, and sparsity checks

First read the **paired, sample-balanced held-out lineage contrast** on the two blinded graphs. Keep the two directions separate: a stronger CD8 axis alone doesn't rescue CD4. Then look at the helper-associated markers, Cd8b1 and all four Foxp3 companion endpoints, followed by depth and fragmentation. Focus on changes across samples rather than pooled cell counts. The atlas's original T cell 1/2 assignments and per-sample state measurements help distinguish a state that varies *within* lineages from a lack of lineage evidence.

UMAPs are illustrations. Refer to original-count dotplots and paired scores before interpreting a visual gap. No detected transcript does not mean the protein or lineage is absent. A marker-blinded graph can still use correlated biology and the common full-library normalization; this is an internal feature test, not independent animal, protein or histology validation.

In [ ]:
try:
    REVIEW = review.build_reports(PLAN, BRANCHES, RUN_ROOT, max_plot_cells=60000)
    _full=Path(REVIEW["index_path"]).resolve().parent
    DISPLAY_DIR=PUBLISH_DIR/PLAN["plan_id"][:12]/Path(RUN_ROOT).name
    DISPLAY_DIR.mkdir(parents=True,exist_ok=True)
    # The report contains summaries and figures. Keep model matrices and graph arrays on the original volume.
    if _full!=DISPLAY_DIR.resolve():
        shutil.copytree(_full,DISPLAY_DIR,dirs_exist_ok=True,ignore=shutil.ignore_patterns("*.h5ad","*.npz","*.npy","*.parquet","*.zip"))
    show_file(DISPLAY_DIR/Path(REVIEW["index_path"]).name,"Open full 08W review:")
    print("Full review on EC2:",_full)
    deltas = pd.read_csv(DISPLAY_DIR / "paired_delta_summary.csv")
    display(Markdown("**Coreceptor-blind test · positive change means stronger than its matched control**"))
    display(deltas.loc[deltas.contrast.eq("anchorblind_state_test") & deltas.endpoint.isin(["exclusive_CD4","exclusive_CD8","balanced_exclusive_anchors","Cd4_to_Zbtb7b","Cd4_to_Cd40lg","Cd8a_to_Cd8b1"])])
    plot = DISPLAY_DIR / "figures" / "paired_changes.png"
    if plot.is_file(): display(Image(filename=str(plot),width=1000))
except (Exception, KeyboardInterrupt) as e:
    diagnostics(e)
    raise

## 4 · Send back the small result ZIP

The result bundle is for review, not a replacement for the completed run. Keep the full models and original 08V analysis on the EC2 volume.

In [ ]:
try:
    LOCAL_RETURN = DISPLAY_DIR / "08W_return_to_chat.zip"
    review.make_return_bundle(PLAN, RUN_ROOT, destination=LOCAL_RETURN)
    show_file(LOCAL_RETURN,"Download this result ZIP to return:")
    print(f"Notebook elapsed: {(time.monotonic()-_started)/60:.1f} min | {LOCAL_RETURN}")
except (Exception, KeyboardInterrupt) as e:
    diagnostics(e)
    raise

### How to decide

- **Lineage improves** if CD4- and CD8-supported cells have more same-lineage neighbors relative to opposite-lineage neighbors on the masked **blind** graph than on its blind control, consistently across samples, with compatible helper/CD8 companion evidence. Lower association with depth or prettier clusters alone is insufficient.
- **Rare filtering helps** if the ≥500-cell selected z-score branch reduces fragmentation relative to the full selected z-scored panel without losing the depth-adjusted Foxp3 companion signal. This comparison is independent of the state masking.
- **State persists:** if cells still organize by stress or exhaustion after direct masking, that can be correlated biology; it does not prove the algorithm is broken or the signal is contamination. If masking erases a condition-associated state without an independent lineage gain, retain the unmasked embedding. Preserve raw readouts of state for the study.
- **Weak evidence on both blinded graphs:** report cells with observed lineage support separately from unresolved cells. For abundance inference across treatment conditions, use mouse/sample as the replicate and seek independent spatial or CD4/CD8 protein validation where practical.

**Biological context:** [in situ T-cell stress in untreated and checkpoint-treated human tumors](https://www.nature.com/articles/s41591-023-02371-y); [mouse CD4 helper program and CD40 ligand](https://pmc.ncbi.nlm.nih.gov/articles/PMC3392968/); [exhaustion in mouse CD4 cells in the context of cancer](https://pmc.ncbi.nlm.nih.gov/articles/PMC12184148/). None establishes which signal is dominant in this dataset.